In [1]:
import arcpy
import os
import json

def create_raster_catalog_arcpy(input_folders, output_geojson):
    """
    Scans folders for rasters using ArcPy to handle ECW/JP2 formats.
    If a raster has no CRS, assigns EPSG:2100 (Greek Grid).
    Projects bounds to WGS84 and saves as GeoJSON.
    """
    valid_extensions = ('.tif', '.tiff', '.jp2','.jpg','.JPG', '.ecw', '.img', '.sid')

    # Define WGS84 Spatial Reference (EPSG:4326) for GeoJSON
    sr_wgs84 = arcpy.SpatialReference(4326)
    
    # Define fallback Spatial Reference (EPSG:2100 - Greek Grid)
    sr_greek = arcpy.SpatialReference(2100)

    geojson_output = {
        "type": "FeatureCollection",
        "features": []
    }

    print(f"Starting scan of {len(input_folders)} folder(s) using ArcPy...")
    count = 0

    for folder in input_folders:
        for root, dirs, files in os.walk(folder):
            for file in files:
                if file.lower().endswith(valid_extensions):
                    file_path = os.path.join(root, file)

                    try:
                        # 1. Check if valid raster and get properties
                        try:
                            desc = arcpy.Describe(file_path)
                        except Exception:
                            continue

                        if desc.dataType != 'RasterDataset':
                            continue

                        # 2. Check for Spatial Reference
                        # Default to the raster's own SR
                        source_sr = desc.spatialReference
                        crs_source_name = "Unknown - Assumed EPSG:2100"

                        # If missing or unknown, define it as EPSG:2100
                        if source_sr is None or source_sr.name == "Unknown":
                            print(f"No CRS found for: {file}. Assuming EPSG:2100.")
                            
                            # We don't need to permanently 'Define Projection' on the file 
                            # if we just want to process the extent correctly in memory.
                            # We simply treat the extent coordinates as belonging to EPSG:2100.
                            source_sr = sr_greek
                        else:
                            crs_source_name = source_sr.name

                        # 3. Create a Polygon from the raster's extent
                        extent = desc.extent

                        # Create a list of points (Counter-clockwise)
                        pts = [
                            arcpy.Point(extent.XMin, extent.YMin),
                            arcpy.Point(extent.XMax, extent.YMin),
                            arcpy.Point(extent.XMax, extent.YMax),
                            arcpy.Point(extent.XMin, extent.YMax),
                            arcpy.Point(extent.XMin, extent.YMin)
                        ]

                        # Create polygon geometry using the determined Source SR
                        poly = arcpy.Polygon(arcpy.Array(pts), source_sr)

                        # 4. Project the polygon to WGS84
                        poly_wgs84 = poly.projectAs(sr_wgs84)

                        # 5. Get the bounding box of the PROJECTED polygon
                        wgs_ext = poly_wgs84.extent

                        coordinates = [[
                            [wgs_ext.XMin, wgs_ext.YMin],
                            [wgs_ext.XMax, wgs_ext.YMin],
                            [wgs_ext.XMax, wgs_ext.YMax],
                            [wgs_ext.XMin, wgs_ext.YMax],
                            [wgs_ext.XMin, wgs_ext.YMin]
                        ]]

                        # 6. Append to Feature Collection
                        feature = {
                            "type": "Feature",
                            "properties": {
                                "filename": file,
                                "filepath": file_path,
                                "crs_source": crs_source_name
                            },
                            "geometry": {
                                "type": "Polygon",
                                "coordinates": coordinates
                            }
                        }

                        geojson_output["features"].append(feature)
                        count += 1
                        print(f"Processed: {file}")

                    except Exception as e:
                        print(f"Error processing {file}: {str(e)}")

    # Write the final file
    with open(output_geojson, 'w') as f:
        json.dump(geojson_output, f, indent=2)

    print(f"\nSuccess! Processed {count} rasters.")
    print(f"Output saved to: {output_geojson}")


In [2]:
my_folders = [r"C:\workspace\Freelance life\Pantelis Karapatsios\ArcPy ortho automation\Data\Structured folders\5ARIA_GEO_CORRECT"]
# 2. Output filename
output_file = r"C:\workspace\Freelance life\Pantelis Karapatsios\ArcPy ortho automation\image_catalog_5aria_full.geojson"
create_raster_catalog_arcpy(my_folders, output_file)

Starting scan of 1 folder(s) using ArcPy...
Processed: 7657_6.tif
Processed: 7657_8.tif
Processed: 7658_5.tif
Processed: 7658_7.tif
Processed: 7658_8.tif
Processed: 7665_7.tif
Processed: 7665_8.tif
Processed: 7666_2.tif
Processed: 7666_3.tif
Processed: 7666_4.tif
Processed: 7666_5.tif
Processed: 7666_6.tif
Processed: 7666_7.tif
Processed: 7666_8.tif
Processed: 7667_1.tif
Processed: 7667_2.tif
Processed: 7667_3.tif
Processed: 7667_5.tif
Processed: 7668_1.tif
Processed: 7668_2.tif
Processed: 7675_1.tif
Processed: 7675_2.tif
Processed: 7675_3.tif
Processed: 7675_4.tif
Processed: 7676_1.tif
Processed: 7676_2.tif
Processed: 6554_4.tif
Processed: 6554_6.tif
Processed: 6554_8.tif
Processed: 6555_1.tif
Processed: 6555_2.tif
Processed: 6555_3.tif
Processed: 6555_4.tif
Processed: 6555_5.tif
Processed: 6555_6.tif
Processed: 6555_7.tif
Processed: 6555_8.tif
Processed: 6556_3.tif
Processed: 6556_5.tif
Processed: 6556_7.tif
Processed: 6556_8.tif
Processed: 6557_7.tif
Processed: 6564_2.tif
Processed: